In [1]:
import os
import json
import requests
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    raise ValueError("GROQ_API_KEY not found. Check your .env file.")

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=api_key,
    temperature=0
)

print("Groq model initialized successfully.")

Groq model initialized successfully.


In [2]:
@tool
def get_weather(city: str) -> str:
    """Get the current temperature in Celsius and weather conditions for a city."""

    geocoding_url = "https://geocoding-api.open-meteo.com/v1/search"

    geo_response = requests.get(
        geocoding_url,
        params={
            "name": city,
            "count": 10,
            "language": "en",
            "format": "json"
        },
        timeout=20
    )
    geo_response.raise_for_status()

    geo_data = geo_response.json()
    results = geo_data.get("results", [])

    if not results:
        return json.dumps({
            "error": f"City not found: {city}"
        })

    # Prefer an exact city-name match when available
    match = next(
        (
            item for item in results
            if item["name"].casefold() == city.strip().casefold()
        ),
        results[0]
    )

    latitude = match["latitude"]
    longitude = match["longitude"]
    city_name = match["name"]
    country = match.get("country", "")

    weather_url = "https://api.open-meteo.com/v1/forecast"

    weather_response = requests.get(
        weather_url,
        params={
            "latitude": latitude,
            "longitude": longitude,
            "current": "temperature_2m,weather_code",
            "timezone": "auto"
        },
        timeout=20
    )
    weather_response.raise_for_status()

    weather_data = weather_response.json()
    current = weather_data.get("current")

    if not current:
        return json.dumps({
            "error": "Current weather data is unavailable."
        })

    weather_codes = {
        0: "Clear sky",
        1: "Mainly clear",
        2: "Partly cloudy",
        3: "Overcast",
        45: "Fog",
        48: "Depositing rime fog",
        51: "Light drizzle",
        53: "Moderate drizzle",
        55: "Dense drizzle",
        61: "Slight rain",
        63: "Moderate rain",
        65: "Heavy rain",
        71: "Slight snow",
        73: "Moderate snow",
        75: "Heavy snow",
        80: "Rain showers",
        81: "Moderate rain showers",
        82: "Violent rain showers",
        95: "Thunderstorm",
        96: "Thunderstorm with hail",
        99: "Thunderstorm with heavy hail"
    }

    code = current["weather_code"]

    result = {
        "city": city_name,
        "country": country,
        "temperature_celsius": current["temperature_2m"],
        "conditions": weather_codes.get(
            code, f"Unknown weather code: {code}"
        ),
        "observed_at": current["time"],
        "source": "Open-Meteo"
    }

    return json.dumps(result)

In [3]:
@tool
def recommend_outfit(temperature: float, conditions: str) -> str:
    """Recommend practical clothing based on temperature in Celsius and current weather conditions."""

    conditions_lower = conditions.lower()

    if temperature >= 30:
        outfit = (
            "Wear lightweight, breathable clothing. "
            "Consider a hat and sunscreen when outdoors."
        )
    elif temperature >= 20:
        outfit = (
            "Wear comfortable, lightweight clothing. "
            "Carry a light layer if needed."
        )
    elif temperature >= 10:
        outfit = (
            "Wear long sleeves and consider a light jacket."
        )
    else:
        outfit = (
            "Wear warm layers and a suitable jacket."
        )

    if any(
        word in conditions_lower
        for word in ["rain", "drizzle", "thunderstorm"]
    ):
        outfit += " Carry an umbrella or raincoat and wear suitable footwear."

    if any(
        word in conditions_lower
        for word in ["fog", "rime"]
    ):
        outfit += " Consider a visible outer layer when walking outdoors."

    return (
        f"Temperature: {temperature}°C\n"
        f"Conditions: {conditions}\n"
        f"Outfit recommendation: {outfit}"
    )


print("Both tools created successfully.")
print("Weather tool:", get_weather.name)
print("Outfit tool:", recommend_outfit.name)

Both tools created successfully.
Weather tool: get_weather
Outfit tool: recommend_outfit


In [4]:
tools = [
    get_weather,
    recommend_outfit
]

tool_map = {
    "get_weather": get_weather,
    "recommend_outfit": recommend_outfit
}

llm_with_tools = llm.bind_tools(tools)

print("Tools bound to the model:")
for item in tools:
    print("-", item.name)

Tools bound to the model:
- get_weather
- recommend_outfit


In [5]:
city = "Mumbai"

user_query = (
    f"What should I wear today in {city}? "
    f"First call get_weather for {city} only. "
    "Do not recommend an outfit until the weather result is returned."
)

messages = [HumanMessage(content=user_query)]

first_response = llm_with_tools.invoke(messages)

print("User query:")
print(user_query)

print("\nFirst-round tool calls:")
for call in first_response.tool_calls:
    print("Tool:", call["name"])
    print("Arguments:", call["args"])

if not any(
    call["name"] == "get_weather"
    for call in first_response.tool_calls
):
    raise RuntimeError(
        "The model did not call get_weather. Rerun this cell."
    )

User query:
What should I wear today in Mumbai? First call get_weather for Mumbai only. Do not recommend an outfit until the weather result is returned.

First-round tool calls:
Tool: get_weather
Arguments: {'city': 'Mumbai'}


In [6]:
messages.append(first_response)

weather_data = None

for call in first_response.tool_calls:
    tool_name = call["name"]

    if tool_name != "get_weather":
        raise RuntimeError(
            f"Expected get_weather in round one, got {tool_name}."
        )

    result = get_weather.invoke(call["args"])
    parsed_result = json.loads(result)

    print("\nFetched weather data:")
    print(json.dumps(parsed_result, indent=2))

    if "error" in parsed_result:
        raise RuntimeError(parsed_result["error"])

    weather_data = parsed_result

    messages.append(
        ToolMessage(
            content=result,
            tool_call_id=call["id"]
        )
    )

if weather_data is None:
    raise RuntimeError("Weather data was not retrieved.")

print("\nWeather retrieval completed.")


Fetched weather data:
{
  "city": "Mumbai",
  "country": "India",
  "temperature_celsius": 30.7,
  "conditions": "Clear sky",
  "observed_at": "2026-10-22T23:30",
  "source": "Open-Meteo"
}

Weather retrieval completed.


In [7]:
second_prompt = HumanMessage(
    content=(
        "Now use the weather data returned by get_weather. "
        "Call recommend_outfit exactly once. "
        "Copy temperature and conditions directly from the fetched data; "
        "do not invent or change either value.\n\n"
        f"Fetched weather data:\n{json.dumps(weather_data)}"
    )
)

messages.append(second_prompt)

second_response = llm_with_tools.invoke(messages)

print("Second-round tool calls:")
for call in second_response.tool_calls:
    print("Tool:", call["name"])
    print("Arguments:", call["args"])

if not any(
    call["name"] == "recommend_outfit"
    for call in second_response.tool_calls
):
    raise RuntimeError(
        "The model did not call recommend_outfit in round two."
    )

Second-round tool calls:
Tool: recommend_outfit
Arguments: {'conditions': 'Clear sky', 'temperature': 30.7}


In [8]:
messages.append(second_response)

outfit_result = None
outfit_args = None

for call in second_response.tool_calls:
    tool_name = call["name"]

    if tool_name != "recommend_outfit":
        raise RuntimeError(
            f"Unexpected second-round tool call: {tool_name}"
        )

    outfit_args = call["args"]
    outfit_result = recommend_outfit.invoke(outfit_args)

    print("Outfit tool arguments:")
    print(json.dumps(outfit_args, indent=2))

    print("\nOutfit tool result:")
    print(outfit_result)

    messages.append(
        ToolMessage(
            content=str(outfit_result),
            tool_call_id=call["id"]
        )
    )

if outfit_result is None:
    raise RuntimeError("Outfit recommendation was not generated.")

expected_temperature = weather_data["temperature_celsius"]
expected_conditions = weather_data["conditions"]

temperature_matches = (
    float(outfit_args["temperature"]) == float(expected_temperature)
)

conditions_match = (
    outfit_args["conditions"].strip().casefold()
    == expected_conditions.strip().casefold()
)

print("\nTemperature reused correctly:", temperature_matches)
print("Conditions reused correctly:", conditions_match)

Outfit tool arguments:
{
  "conditions": "Clear sky",
  "temperature": 30.7
}

Outfit tool result:
Temperature: 30.7°C
Conditions: Clear sky
Outfit recommendation: Wear lightweight, breathable clothing. Consider a hat and sunscreen when outdoors.

Temperature reused correctly: True
Conditions reused correctly: True


In [9]:
final_response = llm.invoke(messages)

print("Final outfit recommendation:")
print(final_response.content)

Final outfit recommendation:
Based on the current weather in Mumbai—30.7 °C with clear skies—you’ll want to stay cool and protected from the sun. Wear lightweight, breathable clothing such as a short‑sleeve shirt or a loose‑fitting top with shorts or a light skirt. Choose natural fabrics like cotton or linen. Add a wide‑brimmed hat or cap, sunglasses, and apply sunscreen if you’ll be outdoors. Comfortable sandals or breathable shoes are ideal, and you might keep a light scarf or shawl handy for any indoor air‑conditioning. Enjoy your day!


In [10]:
assert weather_data is not None
assert outfit_result is not None
assert final_response.content.strip()

assert temperature_matches, (
    "The model did not reuse the fetched temperature exactly."
)

assert conditions_match, (
    "The model did not reuse the fetched weather conditions exactly."
)

print("Validation passed.")
print("Current weather was fetched from Open-Meteo.")
print("The model made a second-round outfit tool call.")
print("The fetched weather values were reused correctly.")
print("The final recommendation was generated.")

Validation passed.
Current weather was fetched from Open-Meteo.
The model made a second-round outfit tool call.
The fetched weather values were reused correctly.
The final recommendation was generated.


In [11]:
print("Lab 6 - Exercise 4 completed successfully.")
print("Tool 1: get_weather")
print("Tool 2: recommend_outfit")
print("Workflow: Fetch weather -> Return tool result")
print("-> Request outfit tool -> Execute -> Generate final answer")

Lab 6 - Exercise 4 completed successfully.
Tool 1: get_weather
Tool 2: recommend_outfit
Workflow: Fetch weather -> Return tool result
-> Request outfit tool -> Execute -> Generate final answer
